<a href="https://colab.research.google.com/github/gabrielaugustavo/LLM/blob/main/GSI073_aula0_luong_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dadosKepler
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h   # <--- ESSENCIAL

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (B, 1, H)
        encoder_outputs: (B, S, H)

        Retorna:
          context: (B, 1, H)
          attn_weights: (B, 1, S)
        """

        # score = h_t · h_s^T
        # (B, 1, H) x (B, H, S) -> (B, 1, S)
        attn_scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))

        attn_weights = F.softmax(attn_scores, dim=-1)  # normaliza nos steps da source

        # context = soma ponderada
        # (B, 1, S) x (B, S, H) -> (B, 1, H)
        context = torch.bmm(attn_weights, encoder_outputs)

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention()

        # Luong concat: concatena hidden + context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Atenção
            context, attn_w = self.attn(out_t, encoder_outputs)

            # concatenação [out_t ; context]
            combined = torch.cat([out_t, context], dim=-1)

            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, encoder_outputs)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
import numpy as np

def decode_step(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação e retorna os pesos de atenção.
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    """
    # As etapas a seguir são extraídas do método forward do Decoder para um único token
    x_embed = decoder.embed(token)  # (B, 1, E)
    out_t, h = decoder.gru(x_embed, h)   # out_t: (B,1,H)

    # Atenção
    context, attn_w = decoder.attn(out_t, encoder_outputs)

    # concatenação [out_t ; context]
    combined = torch.cat([out_t, context], dim=-1);

    logits = decoder.fc(combined)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h, attn_w


def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder agora retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        attention_weights = [] # Para armazenar os pesos de atenção

        for _ in range(max_len):
            # Agora decode_step retorna os pesos de atenção
            token, h, attn_w = decode_step(model.decoder, token, h, encoder_outputs)
            seq_invertida.append(token.item())
            # Remove todas as dimensões de tamanho 1, para obter (S,) do (1, 1, S)
            attention_weights.append(attn_w.squeeze().cpu().numpy())

        # Converte a lista de arrays numpy em um único array numpy
        attention_weights_matrix = np.array(attention_weights) # Shape (output_len, input_len)

        predicted_sequence = decode(seq_invertida)


        return predicted_sequence, attention_weights_matrix

# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)



total_params = 0
trainable_params = 0

for p in model.parameters():
    total_params += p.numel()
    if p.requires_grad:
        trainable_params += p.numel()

print(f"Total params: {total_params}")
print(f"Trainable params: {trainable_params}")


loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
numero_certos = 0;
for _ in range(100):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")
    if pred == s[::-1]:
      numero_certos += 1
print(f"Acertos: {numero_certos}")



Realização da proximidade letra a letra

# Exercício
Compare o resultado do uso do encoder-decoder com atenção com o encoder-decoder sem atenção.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_attention(input_seq, output_seq, attention_matrix):
    # Remove padding do output_seq para plotagem
    output_seq_cleaned = output_seq.replace(' ', '')
    output_chars = list(output_seq_cleaned)
    input_chars = list(input_seq)

    # Ajusta o tamanho da matriz de atenção se output_seq for mais curto que max_len
    # e também se input_seq for mais curto que max_len (padding no encoder)
    actual_input_len = len(input_chars)
    actual_output_len = len(output_chars)

    # Garante que a matriz de atenção tenha as dimensões corretas para a sequência real
    # Adiciona uma verificação para evitar erro se attention_matrix estiver vazia ou tiver dimensões incorretas
    if attention_matrix.shape[0] < actual_output_len or attention_matrix.shape[1] < actual_input_len:
        print("A matriz de atenção não tem o tamanho esperado para a plotagem.\nVerifique se o comprimento da sequência de saída ou entrada está correto.")
        return

    attention_matrix_clipped = attention_matrix[:actual_output_len, :actual_input_len]

    plt.figure(figsize=(actual_input_len + 1, actual_output_len + 1))
    sns.heatmap(attention_matrix_clipped, cmap='viridis', annot=True, fmt=".2f",
                        xticklabels=input_chars, yticklabels=output_chars)
    plt.xlabel('Sequência de Entrada')
    plt.ylabel('Sequência de Saída')
    plt.title('Matriz de Atenção')
    plt.show()

### Exemplo de Visualização da Matriz de Atenção para uma Entrada Específica

In [ ]:
example_seq = "abcd"
# A função predict (definida anteriormente) agora retorna a sequência invertida e a matriz de atenção
predicted_seq, attn_matrix = predict(model, example_seq, max_len=len(example_seq))

print(f"Input: {example_seq}")
print(f"Predicted: {predicted_seq}")

plot_attention(example_seq, predicted_seq, attn_matrix)